# TCC · EDA de Marketing Mix Modeling com Google Meridian

**Dados simulados oficiais, foco GEO × TEMPO.** Não há estimação posterior, ROI ou recomendação de orçamento.

Execute de cima para baixo em runtime novo Python 3.12+ (CPU). A primeira execução instala dependências; isso pode levar alguns minutos. Se o Colab já tinha versões diferentes de NumPy/TensorFlow carregadas, reinicie a sessão após instalar e execute novamente.

O bootstrap clona o repositório público quando necessário, preserva instalações compatíveis e salva tudo na pasta do projeto. A fonte de dados e a API estão fixadas por commit/checksum. Para uma cópia histórica exata do projeto, configure `REPO_REF` com o commit da entrega. Os dados não possuem nomes de canais reais ou localização geográfica real.


In [1]:
# Execute esta célula primeiro. Em Colab novo, usa somente CPU.
import os, sys, subprocess, importlib.util, importlib.metadata
from pathlib import Path

REPO_URL = "https://github.com/thiagossilva17/MMM_Google_Meridian_Project.git"
REPO_REF = "main"  # Para repetir uma execução publicada, substitua pelo SHA do commit.
IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
cwd = Path.cwd()
candidates = [cwd, *cwd.parents]
PROJECT_ROOT = next((p for p in candidates if (p / "src/config.py").exists()), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = cwd / "MMM_Google_Meridian_Project"
    if not PROJECT_ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "checkout", REPO_REF], check=True)
    elif not (PROJECT_ROOT / "src/config.py").exists():
        raise RuntimeError("Diretório de destino existe, mas não contém o projeto esperado.")

required = ["meridian", "pandas", "numpy", "scipy", "matplotlib", "statsmodels", "tabulate"]
missing = any(importlib.util.find_spec(package) is None for package in required)
try:
    wrong_meridian = importlib.metadata.version("google-meridian") != "2.1.0"
except importlib.metadata.PackageNotFoundError:
    wrong_meridian = True
if missing or wrong_meridian:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(PROJECT_ROOT / "requirements-colab.txt")], check=True)

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / "outputs/.mplcache"))
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
from src.config import SEED, ROOT
from src.data import load_panel
from src.reporting import show
from src.provenance import verify_checkout, verify_meridian_origin
print("Código:",verify_checkout(PROJECT_ROOT, REPO_REF))
print("Meridian:",verify_meridian_origin())
import numpy as np
np.random.seed(SEED)
geo_time_panel = load_panel()
print(f"Colab: {IN_COLAB}; projeto: {ROOT}; painel: {geo_time_panel.shape}")


Código: {'requested_ref': 'main', 'resolved_commit': 'ac9960a3a86e7af07a507121043c6914fcbc4195', 'dirty': True}
Meridian: {'url': 'https://github.com/google/meridian.git', 'vcs_info': {'commit_id': '02111531f8661373aa7b6ba31c316c67d72d1dd2', 'requested_revision': '02111531f8661373aa7b6ba31c316c67d72d1dd2', 'vcs': 'git'}}
Colab: False; projeto: /workspace/scratch/5d823022ed95/tcc-revision; painel: (6240, 19)


O manifesto verifica código, raw, ambiente, conclusão das etapas e hashes dos outputs; resultados incompatíveis são recalculados.

In [2]:
from src.provenance import require_stages
try:
    require_stages(range(2))
except RuntimeError as stale:
    print('Recalculando etapas anteriores:',stale)
    from src.eda import data_audit,general,temporal,relationships
    from src.geo_analysis import geo_analysis,media_geo
    from src.official_eda import official_eda
    for analysis in [data_audit,general,temporal,geo_analysis,media_geo,relationships,official_eda][:2]:
        analysis(geo_time_panel)


## 02 — Como o mercado e seus controles evoluem no tempo?

### Pergunta
Como o mercado e seus controles evoluem no tempo?

### Motivação
Choques comuns, tendência e sazonalidade podem criar associações que se parecem com resposta à mídia.

### Método
Séries nacionais e média móvel de 13 semanas, STL anual aproximado de 52 semanas, ACF, diferenças e picos. Para cada canal: mídia, gasto, CPMU e z-score versus KPI. Controles/Promo usam média ponderada por população; orgânico usa soma.

### Interpretação antes de olhar os resultados
Três ciclos anuais oferecem evidência limitada de sazonalidade. O z-score serve para comparar forma, não unidades ou tamanho de efeito. Mudanças de nível são descritivas, sem teste confirmatório de quebra.

**Dependências:** painel validado e funções em `src/eda.py`. O código reutilizável está comentado e pode ser aberto para inspecionar as fórmulas. Todos os resultados são calculados na execução, não preenchidos manualmente.


In [3]:
from src.eda import temporal
report_02 = temporal(geo_time_panel)
show(report_02,part="summary")

### Resultado observado e conclusão parcial

O máximo nacional ocorre em 2023-05-15 (491,287,369) e o mínimo em 2022-09-19 (340,283,293). A média móvel de 13 semanas suaviza a série sem antecipar valores futuros; é apenas visual.

A decomposição STL usa período 52 como aproximação anual para dados semanais. Há apenas 3.0 ciclos: tendência, sazonalidade e mudanças de nível não são evidência confirmatória de mecanismos de negócio. Nenhum teste formal de quebra foi feito. As maiores mudanças e a ACF foram exportadas.

Todos os canais têm gráficos separados de impressões, gasto, CPMU e comparação padronizada com KPI. Controles e Promo são agregados por média ponderada por população porque somar índices sintéticos não tem interpretação. A mídia orgânica é somada. Existem 0 variáveis constantes entre os controles/tratamentos apresentados. Valores próximos de constantes devem ser avaliados na sua escala, sem corte universal.

**Trajetória:** tendência STL de 413,074,059 a 429,184,890 (+3.90%). Maior mudança semanal em 2023-03-06: -101,396,624. ACF nos lags 1/13/52: 0.008/0.090/0.032. Forças descritivas sazonal/tendência: 0.503/0.034, calculadas por 1−Var(resíduo)/Var(componente+resíduo), truncadas em zero; não são parcelas aditivas ou testes.

competitor_sales_control: amplitude [-4.826,3.925], desvio 1.210, 6240 valores distintos; maior mudança nacional ponderada em 2022-05-16 (-2.382). Não é quase constante na escala observada; CV não é interpretável.

sentiment_score_control: amplitude [-4.230,4.320], desvio 1.175, 6239 valores distintos; maior mudança nacional ponderada em 2021-12-13 (+2.357). Não é quase constante na escala observada; CV não é interpretável.

Promo: amplitude [0.000,4.801], desvio 0.694, 3319 valores distintos; maior mudança nacional ponderada em 2023-08-21 (-1.133). Não é quase constante na escala observada; CV não é interpretável.

### O que ainda NÃO podemos concluir

As relações são descritivas. Não estimamos ROI, contribuição incremental ou efeitos causais.
Os dados são simulados: padrões não descrevem um mercado real e não validam, por si, a identificação de um MMM.

### CHECKPOINT

- **O que descobrimos?** **Trajetória:** tendência STL de 413,074,059 a 429,184,890 (+3.90%). Maior mudança semanal em 2023-03-06: -101,396,624. ACF nos lags 1/13/52: 0.008/0.090/0.032. Forças descritivas sazonal/tendência: 0.503/0.034, calculadas por 1−Var(resíduo)/Var(componente+resíduo), truncadas em zero; não são parcelas aditivas ou testes.
- **Quais problemas apareceram?** Somente três ciclos; STL/ACF não confirmam sazonalidade de negócio nem quebras causais.
- **O que falta investigar?** Heterogeneidade entre geos e decomposição within/between.
- **Podemos avançar?** Podemos prosseguir com a investigação exploratória após revisar estes achados; isso não aprova automaticamente a modelagem.


### Evidências e leitura das figuras
Tabelas pequenas são exibidas integralmente. As sínteses selecionam os casos relevantes; os CSVs mantêm os dados completos.

In [4]:
show(report_02,part="evidence")

**control_temporal_summary** — 4 linhas; CSV completo em `outputs/tables/`.

,variable,unique,range,n,mean,median,std,variance,min,max,p01,p05,p25,p50,p75,p95,p99,iqr,cv,zeros,zero_share,cv_interpretation
0,competitor_sales_control,6240,8.750317e+00,6240,-0.044412,-0.050346,1.210067,1.464262e+00,-4.825634,3.924682e+00,-2.819445,-1.995962,-0.872736,-0.050346,0.743633,1.987068e+00,2.869632e+00,1.616369,NaN,0,0.000000,não interpretável: índice centrado/tratamento ...
1,sentiment_score_control,6239,8.550651e+00,6240,-0.033675,-0.036433,1.175478,1.381749e+00,-4.230392,4.320259e+00,-2.818955,-1.940917,-0.808085,-0.036433,0.739391,1.891985e+00,2.756426e+00,1.547475,NaN,0,0.000000,não interpretável: índice centrado/tratamento ...
2,Promo,3319,4.800555e+00,6240,0.492528,0.084382,0.694224,4.819467e-01,0.000000,4.800555e+00,0.000000,0.000000,0.000000,0.084382,0.832968,1.962645e+00,2.697505e+00,0.832968,NaN,2922,0.468269,não interpretável: índice centrado/tratamento ...
3,Organic_channel0_impression,4164,5.801781e+06,6240,532466.670513,269958.000000,696134.901248,4.846038e+11,0.000000,5.801781e+06,0.000000,0.000000,0.000000,269958.000000,799699.250000,1.971783e+06,3.005490e+06,799699.250000,1.307377,2075,0.332532,razão descritiva; verificar domínio


**national_timeseries** — 156 linhas; CSV completo em `outputs/tables/`.

Amostra das primeiras 12 linhas; sínteses selecionadas abaixo e CSV integral disponível.

,conversions,Channel0_impression,Channel1_impression,Channel2_impression,Channel3_impression,Channel4_impression,Channel0_spend,Channel1_spend,Channel2_spend,Channel3_spend,Channel4_spend,Organic_channel0_impression,competitor_sales_control,sentiment_score_control,Promo,revenue_derived,revenue_per_conversion,population
time,,,,,,,,,,,,,,,,,,
2021-01-25,3.863009e+08,44340630,10537596,6501889,66738506,53026011,325137.568300,101595.404270,48315.045180,520082.52420,413173.984050,24028870,-0.032438,-0.342020,0.272211,7.731975e+06,0.020015,21696753.31
2021-02-01,3.553039e+08,47391940,30857600,7966320,79000018,27592214,347511.968090,297505.263890,59197.121480,615634.54030,214996.093200,29188267,0.312002,0.807150,0.151264,7.104532e+06,0.019996,21696753.31
2021-02-08,4.383089e+08,28960185,23144749,4637739,68442555,20057461,212357.019450,223143.885196,34462.687700,533361.91650,156285.964472,12051369,-1.243697,-0.152005,0.596395,8.762589e+06,0.019992,21696753.31
2021-02-15,4.186124e+08,38126016,10667077,3029637,70601091,37246623,279567.522900,102843.758830,22513.003350,550183.03800,290222.395240,9595212,-0.019225,-0.377523,0.341224,8.363356e+06,0.019979,21696753.31
2021-02-22,3.557853e+08,46890794,17925961,10486840,81842660,56259530,343837.209200,172828.342330,77926.915300,637786.80240,438369.280950,9288846,0.344244,0.034181,0.366063,7.109799e+06,0.019983,21696753.31
2021-03-01,4.406152e+08,31496378,19099596,3713252,58861970,26886668,230954.220330,184143.627600,27592.895000,458701.94280,209498.541240,13711117,-0.691891,-0.554818,0.371434,8.813679e+06,0.020003,21696753.31
2021-03-08,4.530552e+08,25772939,4734654,7271164,68725846,37615253,188985.826030,45647.895670,54031.470175,535569.55042,293094.728570,13763654,-0.848894,-0.373470,0.091544,9.050774e+06,0.019977,21696753.31
2021-03-15,3.905958e+08,35435485,17853348,8084840,73628817,40915556,259838.598400,172128.262440,60077.836005,573777.62510,318810.394260,17933987,0.454243,0.151022,0.012450,7.804657e+06,0.019981,21696753.31
2021-03-22,3.790501e+08,28273144,27489089,6237490,64406105,56582012,207319.137090,265028.672380,46350.316360,501906.50854,440882.035900,31799726,0.560951,-1.544945,0.928034,7.584097e+06,0.020008,21696753.31


...

[Figura gerada; veja outputs/figures ou execute a célula.]

**Como ler — Organic channel0 impression time:** Compare com os resultados numéricos e a interpretação do capítulo. Unidade e denominador delimitam a leitura; associação não demonstra eficácia.

[Figura gerada; veja outputs/figures ou execute a célula.]

**Como ler — Organic channel0 impression distribution:** Compare com os resultados numéricos e a interpretação do capítulo. Unidade e denominador delimitam a leitura; associação não demonstra eficácia.

[Figura gerada; veja outputs/figures ou execute a célula.]